# Notebook 1 — Data Acquisition

This notebook guides you through obtaining satellite imagery from:
1. **Google Earth Engine** (Sentinel-2, Landsat) — programmatic, server-side
2. **ISRO Bhuvan** — manual download guide for LISS-IV / Cartosat-3
3. **Copernicus Dataspace** — direct Sentinel-2 download

### Prerequisites
```bash
pip install -r requirements.txt
earthengine authenticate  # one-time GEE auth
```

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import json
from pathlib import Path

# Load project configuration
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

print('AOI:', cfg['aoi']['name'])
print('BBox:', cfg['aoi']['bbox'])
print('Baseline:', cfg['dates']['baseline'])
print('Current:', cfg['dates']['current'])
print('Source:', cfg['source']['primary'])

## 1. Google Earth Engine — Sentinel-2

GEE processes data server-side — no large downloads needed for initial analysis.
We pull cloud-free median composites for the baseline and current periods.

**Sentinel-2 resolution**:
- 10 m: RGB (B2,B3,B4) + NIR (B8) → NDVI, EVI
- 20 m: Red-Edge (B5,B6,B7), SWIR (B11,B12) → NDRE, NBR

**Reference**: ESA Sentinel-2 User Guide — [sentinel.esa.int](https://sentinel.esa.int)

In [ ]:
import ee
from src.data.gee_downloader import (
    initialize_gee,
    get_sentinel2_median,
    get_hansen_forest_change,
    get_gedi_canopy_height,
    export_image_to_drive,
)

# Authenticate and initialise GEE
initialize_gee()

bbox = cfg['aoi']['bbox']
max_cloud = cfg['source']['cloud_cover_max']

# Fetch cloud-free median composites
s2_baseline = get_sentinel2_median(
    bbox,
    cfg['dates']['baseline']['start'],
    cfg['dates']['baseline']['end'],
    max_cloud,
)

s2_current = get_sentinel2_median(
    bbox,
    cfg['dates']['current']['start'],
    cfg['dates']['current']['end'],
    max_cloud,
)

print('Baseline bands:', s2_baseline.bandNames().getInfo())
print('Current bands:', s2_current.bandNames().getInfo())

In [ ]:
# Compute NDVI server-side in GEE
def add_ndvi(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return img.addBands(ndvi)

s2_baseline = add_ndvi(s2_baseline)
s2_current  = add_ndvi(s2_current)

# Get mean NDVI for the AOI
aoi = ee.Geometry.Rectangle(bbox)
mean_ndvi_baseline = s2_baseline.select('NDVI').reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=10
).getInfo()
mean_ndvi_current = s2_current.select('NDVI').reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=10
).getInfo()

print(f"Mean NDVI ({cfg['dates']['baseline']['label']}): {mean_ndvi_baseline['NDVI']:.4f}")
print(f"Mean NDVI ({cfg['dates']['current']['label']}):  {mean_ndvi_current['NDVI']:.4f}")
print(f"NDVI change: {mean_ndvi_current['NDVI'] - mean_ndvi_baseline['NDVI']:+.4f}")

In [ ]:
# Export NDVI composites to Google Drive for local analysis
# This creates background tasks — monitor at: https://code.earthengine.google.com/tasks

aoi_name = cfg['aoi']['name'].replace(' ', '_')

task_baseline = export_image_to_drive(
    s2_baseline.select(['B2', 'B3', 'B4', 'B8', 'B5', 'B11', 'NDVI']),
    description=f'{aoi_name}_S2_baseline_{cfg["dates"]["baseline"]["label"]}',
    scale=10,
    bbox=bbox,
)

task_current = export_image_to_drive(
    s2_current.select(['B2', 'B3', 'B4', 'B8', 'B5', 'B11', 'NDVI']),
    description=f'{aoi_name}_S2_current_{cfg["dates"]["current"]["label"]}',
    scale=10,
    bbox=bbox,
)

print('Export tasks started.')
print(f'  Baseline: {task_baseline.status()["description"]}')
print(f'  Current:  {task_current.status()["description"]}')
print()
print('Download the exported GeoTIFFs from Google Drive and place them in:')
print(f'  data/raw/sentinel2/{cfg["dates"]["baseline"]["label"]}/')
print(f'  data/raw/sentinel2/{cfg["dates"]["current"]["label"]}/')

## 2. Hansen Global Forest Change (Reference Layer)

Hansen et al. 2013 provides annual tree cover loss/gain at 30 m resolution
from 2001 to present. Use this as a cross-validation layer.

**Reference**: Hansen, M.C. et al. (2013). *Science*, 342(6160), 850-853.

In [ ]:
hansen = get_hansen_forest_change(bbox)

# Pixel counts for the AOI
stats = {
    k: v.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi, scale=30, maxPixels=1e10,
    ).getInfo()
    for k, v in hansen.items()
}

treecover_px = stats['treecover2000'].get('treecover2000', 0)
loss_px = stats['loss'].get('loss', 0)
gain_px = stats['gain'].get('gain', 0)

# At 30m resolution, each pixel = 900 m² = 0.09 ha
px_ha = 900 / 10_000
print(f'Tree cover in year 2000: {treecover_px * px_ha:,.1f} ha (30m pixels)') 
print(f'Forest loss 2001–2023:   {loss_px * px_ha:,.1f} ha')
print(f'Forest gain 2000–2012:   {gain_px * px_ha:,.1f} ha')

## 3. GEDI LiDAR — Canopy Height

GEDI (Global Ecosystem Dynamics Investigation) provides canopy height estimates
from the ISS. Available for latitudes between 51.6°N and 51.6°S.

**Reference**: Dubayah, R. et al. (2020). *Remote Sensing of Environment*, 251, 112165.

In [ ]:
gedi = get_gedi_canopy_height(bbox)

canopy_height_stats = gedi.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry=aoi, scale=25, maxPixels=1e10,
).getInfo()

print('GEDI Canopy Height (rh98):')
print(f'  Mean:   {canopy_height_stats.get("rh98_mean", "N/A"):.1f} m')
print(f'  StdDev: {canopy_height_stats.get("rh98_stdDev", "N/A"):.1f} m')
print()
print('Note: rh98 = relative height at which 98% of returned LiDAR energy')
print('is captured — a proxy for dominant canopy height.')

## 4. ISRO Data — Manual Download Guide

For Indian-specific high-resolution data (LISS-IV at 5.8 m), follow these steps:

In [ ]:
from src.data.bhuvan_downloader import print_access_guide, recommend_sensor, ISRO_SENSORS

# Print the step-by-step guide
print_access_guide()

In [ ]:
# Recommend the best ISRO sensor for your use case
recommendations = recommend_sensor(
    resolution_needed_m=10,   # we want ≤ 10m for canopy segmentation
    requires_nir=True,        # we need NIR for NDVI
    aoi_size_km2=50,
)
print('Recommended ISRO sensors (best to adequate):')
for name in recommendations:
    s = ISRO_SENSORS[name]
    print(f'  {name:20s} — {s["resolution_m"]} m, {s["revisit_days"]}-day revisit')
    if 'note' in s:
        print(f'    Note: {s["note"]}')

## 5. Data Directory Setup

After downloading images, organise them as follows:
```
data/
  raw/
    sentinel2/
      2016/  ← baseline GeoTIFFs downloaded from GDrive
      2024/  ← current GeoTIFFs
    bhuvan/
      liss4/
        2016/  ← extracted ISRO ZIP files
        2024/
  processed/
    ndvi/
    masks/
```

Run **Notebook 2** to preprocess and compute vegetation indices on local files.

In [ ]:
# Create local directory structure
dirs = [
    'data/raw/sentinel2', 'data/raw/bhuvan/liss4', 'data/raw/landsat',
    'data/processed/ndvi', 'data/processed/masks', 'data/processed/crowns',
    'models', 'results',
]
for d in dirs:
    Path(f'../{d}').mkdir(parents=True, exist_ok=True)
print('Directory structure created.')